# 03. e + Ba⁺: сечения и константы скоростей для 455.403 и 493.408 нм

* A и ветвления — NIST ASD (известны все ветви: 6p ²P₃/₂ → 6s, 5d₅/₂, 5d₃/₂; 6p ²P₁/₂ → 6s,
  5d₃/₂; класс точности B) — `nb.BA_LINES`;
* сечения возбуждения — **считаются здесь** (готовых данных нет в наших источниках):
  модель `ba2` (`models.py`): один валентный электрон над [Xe]; 6s, 5d, 7s, 6d, 8s, 7d |
  6p, 7p, 4f, 5f (17 состояний), пороги — NIST.

Проверки модели: энергии против NIST; A против NIST; времена жизни 5d (E2) против
измеренных (впишите из литературы); связанные (N+1)-состояния = уровни Ba I.

Время (замер на 4 ядрах, J ≤ 3, 8 волн): самая большая матрица 10730, диагонализация одной
волны до 24 мин. Полный расчёт (J ≤ 20) — оценка: часы на 32 ядрах.

**Выход** (папка `ba2`): `sigma_ba2.xlsx` и `rates_ba2.xlsx` (все возбуждения между 17
состояниями, формат Ванга, номера NIST), `ba2_line_sigma.xlsx`, `ba2_k_line.xlsx` (+ .csv).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
ROOT    = Path("/data/files_Otdel120/pydbsr_calc")    # папка установки (большой диск)
REPO    = ROOT / "pydbsr_src"                         # распакованный архив pydbsr (или клон репозитория)
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = ROOT                                        # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = ROOT / "runs" / "crm"                       # результаты расчётов (большой диск)
XE2_RUN = ROOT / "runs" / "xe_plus_v2"                # готовый расчёт e + Xe+ (модель Ванга, J <= 25)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

In [ ]:
JMAX = 20           # сходимость по J проверьте: 15 / 20 / 25
job = nb.Job(RUNS / "ba2", "ba2", stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=30, de=0.005, te=",".join(map(str, TE)))
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

## 1. Энергии мишени

In [ ]:
W = RUNS / "ba2"
print((W / "target_table.txt").read_text())

## 2. A против NIST, времена жизни 5d

In [ ]:
NIST_A = {}
for x in nb.BA_LINES:
    if x["ion"] == "Ba II":
        NIST_A.update(x["A_branches"])
for r in nb.read_csv(W / "transitions_E1.csv"):
    wl = nb.fnum(r["wl_air_nm"])
    hit = [k for k in NIST_A if wl and abs(k - wl) < 0.05]
    if hit:
        a = float(r["A_exp"])
        print(f"{hit[0]:9.3f}  {r['upper_nist']:>14s} -> {r['lower_nist']:<14s}  DBSR {a:.3e}  NIST {NIST_A[hit[0]]:.3e}"
              f"  DBSR/NIST {a / NIST_A[hit[0]]:.2f}  vel/len {r['gauge_ratio']}")
MEASURED_TAU_S = {}        # впишите: {"5d 2D J=3/2": τ (с), "5d 2D J=5/2": τ (с)} со ссылкой
_, tot, _ = nb.branching_from_csv(W / "transitions_E2.csv")
for r in nb.read_csv(W / "transitions_E2.csv"):
    if r["upper_nist"].startswith("5d") and r["lower_nist"].startswith("6s"):
        u = int(r["upper_no"])
        print(f"E2 {r['upper_nist']}: τ = {1 / tot[u]:.1f} с  (изм.: {MEASURED_TAU_S.get(r['upper_nist'], '—')})")

## 3. Связанные состояния e + Ba⁺ = уровни Ba I (отдельный короткий прогон, J ≤ 4)

In [ ]:
jb = nb.Job(RUNS / "ba2_bound", "ba2", stage="bound", jmax=4, cores=CORES, mem=MEM, hd_threads=HD_THREADS)
jb.start()

In [ ]:
jb.status()
lv1 = db.nist.fetch_levels("Ba I")
for b in sorted(nb.read_csv(RUNS / "ba2_bound" / "bound_states.csv"), key=lambda b: float(b["E_cm"]))[:20]:
    e = float(b["E_cm"])                      # над самым нижним связанным состоянием (Ba I 6s2)
    near = min(lv1, key=lambda l: abs(l.energy_cm - e))
    print(f"{b['label']:>14s}  E = {e:9.1f} см-1   ближайший NIST: {near.config} {near.term} J={near.J:g}"
          f"  {near.energy_cm:9.1f}  Δ = {e - near.energy_cm:+7.0f}")

## 4. Сечения линий 455.4, 493.4 и константы скоростей

In [ ]:
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz", partial_waves=False)
tg = db.Target.load(W / "target")
st = {s.name: s for s in tg.states if s.name in cs.names}
name_of = {s.nist_label: n for n, s in st.items()}
TEa = np.array(TE)
sig, lab, k_line, lab_l = {}, {}, {}, {}
fig, ax = plt.subplots(figsize=(7, 4))
for x in [x for x in nb.BA_LINES if x["ion"] == "Ba II"]:
    u = name_of[x["upper"]]
    for i, si in st.items():
        if si.exp_energy_cm >= st[u].exp_energy_cm:
            continue
        E, s = cs.incident_energy(i), cs.sigma(i, u)
        m = np.isfinite(s) & (E > 0)
        key = (si.nist_no, st[u].nist_no, x["wl"])
        sig[key] = (E[m], x["branching"] * s[m])
        lab[key] = f"{si.nist_no}->{st[u].nist_no} ({x['wl']:.3f})"
        # <sigma v> from sigma with a dipole tail above the computed grid (6s/5d -> 6p are E1-allowed);
        # cs.rate would integrate Upsilon only over the grid (too low at high Te)
        k_line[key] = x["branching"] * rates.rates_on_grid(E[m], s[m], TEa, tail="dipole")
        lab_l[key] = lab[key]
        if si.nist_no == 1:
            ax.plot(E[m], x["branching"] * s[m] / 1e-16, lw=0.8, label=f"{x['wl']} нм из 6s")
ax.set_xscale("log"); ax.set_xlabel("E, эВ"); ax.set_ylabel("σ линии, 10⁻¹⁶ см²"); ax.legend(); plt.show()
lv = [l for l in db.nist.fetch_levels("Ba II") if l.no in {s.nist_no for s in st.values()}]
reference.write_xlsx(W / "ba2_line_sigma.xlsx", lv, sig, sheet_of=lambda k: f"{k[2]:.3f}", labels=lab)
reference.write_rates_xlsx(W / "ba2_k_line.xlsx", TEa, k_line, lab_l,
                           note="BR * <sigma v> (cm3/s); sigma: pydbsr model ba2 (this run); BR: NIST ASD")
print("->", W / "ba2_line_sigma.xlsx", W / "ba2_k_line.xlsx")

## 5. Поправка на силу линии 6s → 6p (поляризация остова)

Модель `ba2` без поляризации остова 5p⁶: сила линии S(6s→6p) получается больше
экспериментальной (тест: в 1.3 раза; S из A NIST: S = A·g_u·λ³[Å]/2.026·10¹⁸). Возбуждение
6s → 6p — дипольно разрешённое, при E ≫ порога σ ∝ S, и завышение S переходит в σ почти
целиком. Ниже — отношение S_NIST/S_DBSR и вариант σ и k линий из 6s, умноженных на него
(файлы `*_scaledS`). Это приближение (у порога недипольные вклады и резонансы так не
масштабируются); какой вариант брать — решите по сравнению с экспериментом
**Pace, Hooper, Phys. Rev. A 7, 2033 (1973)**, doi:10.1103/PhysRevA.7.2033 — «Absolute Experimental Cross Sections for the Excitation of Barium Ions by Electron Impact» (абсолютные σ возбуждения Ba⁺)
или другим расчётом. Различие вариантов — оценка неопределённости плотности Ba⁺.

In [ ]:
ratio = {}
for r in nb.read_csv(W / "transitions_E1.csv"):
    if r["lower_nist"].startswith("6s") and r["upper_nist"].startswith("6p"):
        wl = nb.fnum(r["wl_air_nm"])
        A_n = [v for k, v in NIST_A.items() if wl and abs(k - wl) < 0.05]
        if not A_n:
            continue
        g_u = int(r["upper_nist"].split("J=")[1].split("/")[0]) + 1
        S_n = A_n[0] * g_u * (10 * wl) ** 3 / 2.026e18
        ratio[r["upper_nist"]] = S_n / float(r["S"])
        print(f"{r['upper_nist']:>14s} -> 6s  S_DBSR = {float(r['S']):.2f}  S_NIST = {S_n:.2f}"
              f"  S_NIST/S_DBSR = {ratio[r['upper_nist']]:.3f}")
sig_s, k_s = {}, {}
for key in sig:
    x = [y for y in nb.BA_LINES if abs(y["wl"] - key[2]) < 1e-3][0]
    f = ratio.get(x["upper"], 1.0) if key[0] == 1 else 1.0      # только из основного 6s
    sig_s[key] = (sig[key][0], f * sig[key][1])
    k_s[key] = f * k_line[key]
reference.write_xlsx(W / "ba2_line_sigma_scaledS.xlsx", lv, sig_s, sheet_of=lambda k: f"{k[2]:.3f}", labels=lab)
reference.write_rates_xlsx(W / "ba2_k_line_scaledS.xlsx", TEa, k_s, lab_l,
                           note="as ba2_k_line; from 6s scaled by S_NIST/S_DBSR of 6s-6p (no core polarisation in the model)")
print("->", W / "ba2_line_sigma_scaledS.xlsx", W / "ba2_k_line_scaledS.xlsx")

## 6. Сравнение с экспериментом Pace & Hooper (1973)

Pace, Hooper, Phys. Rev. A 7, 2033 (1973) измерили **сечения испускания** линий 455.4 и
493.4 нм при ударе электронов по Ba⁺ 6s (`nb.PACE_HOOPER`, табл. X и XI; погрешность
21–43 %). В сечение испускания входит всё, что даёт фотон линии:
σ_исп = BR · [σ(6s→6p_j) + Σ_h σ(6s→h)·P(h→6p_j)], где P(h→u) — доля распадов уровня h
(7s, 6d, …), проходящих через u (ветвления DBSR E1 этого расчёта), BR — NIST.
Сравниваем то же самое: прямое, с каскадами, с каскадами и поправкой S (раздел 5). Для отношений
расчёт сглаживается гауссианой с шириной `E_SPREAD` (разброс энергии электронов в эксперименте),
иначе у порога точка расчёта попадает на отдельный резонанс. Погрешность эксперимента 21–43 % —
без учёта абсолютной калибровки оптики (их табл. II).

In [ ]:
br_e1, _, _ = nb.branching_from_csv(W / "transitions_E1.csv")
P = nb.cascade_prob(br_e1)
by_no = {s.nist_no: n for n, s in st.items()}
i0 = by_no[1]
E0 = cs.incident_energy(i0)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, x in zip(axs, [x for x in nb.BA_LINES if x["ion"] == "Ba II"]):
    u = name_of[x["upper"]]; un = st[u].nist_no
    direct = np.nan_to_num(cs.sigma(i0, u))
    casc = sum(np.nan_to_num(cs.sigma(i0, by_no[h])) * p.get(un, 0.0)
               for h, p in P.items() if h in by_no and p.get(un, 0.0) > 0)
    f = ratio.get(x["upper"], 1.0)
    m = E0 > 0
    E_SPREAD = 0.5     # эВ, FWHM разброса энергии электронов в эксперименте (уточните по статье); сглаживание DBSR
    def smooth(y):
        g = np.exp(-0.5 * ((E0[m][:, None] - E0[m][None, :]) / (E_SPREAD / 2.3548)) ** 2)
        return (g @ y[m]) / g.sum(axis=1)
    ax.plot(E0[m], x["branching"] * direct[m] / 1e-16, lw=0.7, label="DBSR, прямое")
    ax.plot(E0[m], x["branching"] * (direct + casc)[m] / 1e-16, lw=0.9, label="DBSR + каскады")
    ax.plot(E0[m], x["branching"] * (f * direct + casc)[m] / 1e-16, lw=0.9, ls="--", label="DBSR + каскады, S → NIST")
    ph = nb.PACE_HOOPER[x["wl"]]
    ax.errorbar(ph["E"], ph["sigma"], yerr=ph["sigma"] * ph["err"] / 100, fmt="ko", ms=4, label="Pace, Hooper 1973")
    ax.set_xscale("log"); ax.set_title(f"{x['wl']} нм"); ax.set_xlabel("E, эВ"); ax.set_ylabel("σ испускания, 10⁻¹⁶ см²")
    print(f"\n{x['wl']} нм:   E, эВ   эксп.   DBSR+касц/эксп.   (с поправкой S)/эксп.")
    for e, sp in zip(ph["E"], ph["sigma"]):
        k = np.argmin(abs(E0 - e))
        if E0[k] <= 0 or abs(E0[k] - e) > 0.5:
            continue
        km = int(np.argmin(abs(E0[m] - e)))                      # точка сглаженной кривой
        a = x["branching"] * smooth(direct + casc)[km] / 1e-16
        b = x["branching"] * smooth(f * direct + casc)[km] / 1e-16
        print(f"            {e:6.1f}  {sp:6.1f}   {a / sp:8.2f}            {b / sp:8.2f}")
axs[0].legend(fontsize=7); fig.tight_layout(); plt.show()